# Gradient Boosting Regression - California Housing Dataset

## Step 1: Import Required Libraries

We need to import essential libraries for data manipulation, visualization, and machine learning:
- **numpy**: For numerical operations
- **pandas**: For data manipulation and analysis
- **matplotlib**: For plotting and visualization
- **sklearn**: For machine learning algorithms and datasets

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing

## Step 2: Load the Dataset

We'll use the California Housing dataset from sklearn. This dataset contains information about California housing districts with the goal of predicting the median house value.

**Steps:**
1. Fetch the dataset using `fetch_california_housing()`
2. Convert the data into a pandas DataFrame
3. Add the target variable (Price) as a new column
4. Display the first few rows to understand the data structure

In [ ]:
housing = fetch_california_housing()

df = pd.DataFrame(
    housing.data,
    columns=housing.feature_names
)

df["Price"] = housing.target

df.head()

## Step 3: Data Exploration

Understanding the dataset structure and statistics is crucial before building any model.

In [ ]:
df.shape
df.info()
df.describe()

## Step 4: Check for Missing Values

In [ ]:
df.isnull().sum()

## Step 5: Check for Duplicate Rows

In [ ]:
df.duplicated().sum()

## Step 6: Split Features and Target

In [ ]:
X = df.drop("Price",axis=1)

y = df["Price"]

## Step 7: Data Visualization - Histograms

In [ ]:
df.hist(
    figsize=(14,10),
    bins=30
)

plt.show()

## Step 8: Correlation Analysis

In [ ]:
corr=df.corr()

corr["Price"].sort_values(
    ascending=False
)
import seaborn as sns

plt.figure(figsize=(10,8))

sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm"
)

plt.show()

## Step 9: Outlier Detection - Boxplots

In [ ]:
for col in df.columns:

    plt.figure()

    plt.boxplot(df[col])

    plt.title(col)

    plt.show()

## Step 10: Feature Scaling (Optional for Gradient Boosting)

**Note:** Gradient Boosting is relatively insensitive to feature scaling, but we'll scale for consistency with other models.

**StandardScaler Formula:**
$$z = \frac{x - \mu}{\sigma}$$

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler=StandardScaler()

X_scaled=scaler.fit_transform(X)

X_scaled=pd.DataFrame(
    X_scaled,
    columns=X.columns
)

## Step 11: Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=\
train_test_split(
X_scaled,
y,
test_size=0.2,
random_state=42
)
X_train.shape
X_test.shape

## Gradient Boosting Regression Formula and Concepts

**Gradient Boosting Concept:**

Gradient Boosting is an ensemble learning method that builds models sequentially. Each new model is trained to correct the errors made by the previous models.

**The Boosting Process:**

1. **Initial Prediction**: Start with a simple model (usually the mean of target values)
2. **Calculate Residuals**: Compute the difference between actual and predicted values
3. **Train New Model**: Train a new weak learner (decision tree) on the residuals
4. **Update Prediction**: Add the new model's prediction (scaled by learning rate)
5. **Repeat**: Continue for a specified number of iterations

**Mathematical Formulation:**

At iteration m:

$$F_m(x) = F_{m-1}(x) + \eta \cdot h_m(x)$$

Where:
- **Fₘ(x)** = prediction after m iterations
- **Fₘ₋₁(x)** = prediction after m-1 iterations
- **η (eta)** = learning rate (shrinkage parameter, typically 0.01-0.1)
- **hₘ(x)** = weak learner trained on residuals at iteration m
- **x** = input features

**Residual Calculation:**

$$r_i = y_i - F_{m-1}(x_i)$$

Where:
- **rᵢ** = residual for the i-th sample
- **yᵢ** = actual target value
- **Fₘ₋₁(xᵢ)** = prediction for the i-th sample

**Key Hyperparameters:**
- **n_estimators**: Number of boosting iterations (trees)
- **learning_rate**: Shrinkage parameter (controls contribution of each tree)
- **max_depth**: Maximum depth of each weak learner (typically 3-5)
- **min_samples_split**: Minimum samples required to split a node
- **min_samples_leaf**: Minimum samples in a leaf node
- **subsample**: Fraction of samples to use for fitting each tree (stochastic gradient boosting)
- **loss**: Loss function to optimize ('squared_error', 'absolute_error', etc.)

**Advantages:**
- **High accuracy**: Often achieves state-of-the-art performance
- **Handles non-linearity**: Can capture complex patterns
- **Feature importance**: Provides feature importance scores
- **Flexible**: Can optimize different loss functions
- **No scaling required**: Works with raw features

**Disadvantages:**
- **Prone to overfitting**: Can overfit if not properly regularized
- **Computationally expensive**: Sequential training (can't parallelize)
- **Sensitive to hyperparameters**: Requires careful tuning
- **Less interpretable**: Harder to understand than single models
- **Longer training time**: Due to sequential nature

**Key Concepts:**
- **Weak Learners**: Typically shallow decision trees (stumps)
- **Learning Rate**: Controls how much each tree contributes
- **Shrinkage**: Small learning rate helps prevent overfitting
- **Stochastic GB**: Using subsamples adds regularization

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

## Train Gradient Boosting with Default Parameters

In [ ]:
gb = GradientBoostingRegressor(n_estimators=100, random_state=42)

gb.fit(
    X_train,
    y_train
)

In [ ]:
y_pred = gb.predict(
    X_test
)

In [ ]:
mae = mean_absolute_error(
    y_test,
    y_pred
)

mse = mean_squared_error(
    y_test,
    y_pred
)

rmse = mse**0.5

r2 = r2_score(
    y_test,
    y_pred
)

print("MAE:",mae)
print("MSE:",mse)
print("RMSE:",rmse)
print("R²:",r2)

## Model Information

In [ ]:
print("Number of Estimators (Trees):", gb.n_estimators_)
print("Learning Rate:", gb.learning_rate)
print("Max Depth:", gb.max_depth)

## Feature Importance

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": gb.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print(feature_importance)

# Visualize feature importance
plt.figure(figsize=(10, 6))
plt.barh(feature_importance["Feature"], feature_importance["Importance"])
plt.xlabel("Feature Importance")
plt.ylabel("Features")
plt.title("Gradient Boosting Feature Importance")
plt.tight_layout()
plt.show()

## Visualization: Actual vs Predicted

In [ ]:
plt.figure(figsize=(8,6))

plt.scatter(
    y_test,
    y_pred
)

plt.plot(
    [y_test.min(),y_test.max()],
    [y_test.min(),y_test.max()]
)

plt.xlabel("Actual")

plt.ylabel("Predicted")

plt.title(
    "Gradient Boosting Regression"
)

plt.show()

## Residual Plot

In [ ]:
residuals = y_test-y_pred

plt.figure(figsize=(8,6))

plt.scatter(
    y_pred,
    residuals
)

plt.axhline(y=0)

plt.xlabel(
    "Predicted"
)

plt.ylabel(
    "Residuals"
)

plt.title(
    "Residual Plot"
)

plt.show()

## Hyperparameter Tuning: Number of Estimators

Let's test different numbers of estimators to see how it affects performance.

In [ ]:
n_estimators_list = [50, 100, 200, 500]

results = []

for n in n_estimators_list:
    model = GradientBoostingRegressor(n_estimators=n, random_state=42)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    r2 = r2_score(y_test, pred)
    
    results.append({
        'n_estimators': n,
        'R²': r2
    })

results_df = pd.DataFrame(results)
print(results_df)

## Hyperparameter Tuning: Learning Rate

The learning rate controls how much each tree contributes. Lower values require more trees but often generalize better.

In [ ]:
learning_rates = [0.01, 0.05, 0.1, 0.2]

results = []

for lr in learning_rates:
    model = GradientBoostingRegressor(n_estimators=100, learning_rate=lr, random_state=42)
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    r2 = r2_score(y_test, pred)
    
    results.append({
        'learning_rate': lr,
        'R²': r2
    })

results_df = pd.DataFrame(results)
print(results_df)

## Compare with Random Forest

Let's compare Gradient Boosting with Random Forest to see the performance difference.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
r2_rf = r2_score(y_test, y_pred_rf)

# Gradient Boosting
gb = GradientBoostingRegressor(n_estimators=100, random_state=42)
gb.fit(X_train, y_train)
y_pred_gb = gb.predict(X_test)
r2_gb = r2_score(y_test, y_pred_gb)

print("Comparison Results:")
print(f"Random Forest: R² = {r2_rf:.4f}")
print(f"Gradient Boosting: R² = {r2_gb:.4f}")
print(f"Difference: {(r2_gb - r2_rf):.4f}")

## Summary

Gradient Boosting Regression provides:
- **High accuracy**: Often achieves state-of-the-art performance
- **Sequential learning**: Each model corrects errors from previous ones
- **Flexibility**: Can optimize different loss functions
- **Feature importance**: Identifies most important features
- **No scaling required**: Works with raw features

**Key advantages over Random Forest:**
- Often achieves better accuracy
- Can optimize specific loss functions
- Better at capturing complex patterns

**Key considerations:**
- **Prone to overfitting**: Requires careful hyperparameter tuning
- **Sequential training**: Slower than Random Forest (can't parallelize)
- **Sensitive to hyperparameters**: Learning rate and n_estimators need tuning
- **Longer training time**: Due to sequential nature

**Best practices:**
- Use cross-validation for hyperparameter tuning
- Start with default parameters and tune gradually
- Use smaller learning rates with more estimators for better generalization
- Consider early stopping to prevent overfitting
- Monitor training vs validation performance